

1. load all TMF tiles
2. one by one, iterate through and extract samples if 


In [1]:
import os
import sys

import pandas as pd
import geopandas as gpd
import rasterio
import numpy as np

sys.path.append('../')

from utils import get_tiles_as_gdf

In [2]:
patch_rows, patch_cols = 33, 33 #based on how i extracted training samples

min_px_forest_loss = 10 #I can also try with less
target_years = [2021, 2022, 2023]

In [3]:

forestloss_dir = '../../Vancutsem_yearloss'
forestloss_tiles = get_tiles_as_gdf(forestloss_dir)

In [4]:
forestloss_tiles

,geometry,filename
0,"POLYGON ((-100.00000 20.00000, -100.00000 30.0...",../../Vancutsem_yearloss/tile_N30_W110.tif
1,"POLYGON ((-90.00000 20.00000, -90.00000 30.000...",../../Vancutsem_yearloss/tile_N30_W100.tif
2,"POLYGON ((-80.00000 20.00000, -80.00000 30.000...",../../Vancutsem_yearloss/tile_N30_W90.tif
3,"POLYGON ((-70.00000 20.00000, -70.00000 30.000...",../../Vancutsem_yearloss/tile_N30_W80.tif
4,"POLYGON ((80.00000 20.00000, 80.00000 30.00000...",../../Vancutsem_yearloss/tile_N30_E70.tif
...,...,...
81,"POLYGON ((160.00000 -30.00000, 160.00000 -20.0...",../../Vancutsem_yearloss/tile_S20_E150.tif
82,"POLYGON ((170.00000 -30.00000, 170.00000 -20.0...",../../Vancutsem_yearloss/tile_S20_E160.tif
83,"POLYGON ((-50.00000 -40.00000, -50.00000 -30.0...",../../Vancutsem_yearloss/tile_S30_W60.tif
84,"POLYGON ((150.00000 -40.00000, 150.00000 -30.0...",../../Vancutsem_yearloss/tile_S30_E140.tif


In [5]:
target_years

[2021, 2022, 2023]

In [6]:
x = forestloss_tiles.filename.iloc[11]

In [7]:
x

'../../Vancutsem_yearloss/tile_N20_W90.tif'

In [8]:
with rasterio.open(x) as source:
    data = source.read(1)

In [9]:
patches_per_tile_min5pix = []

for idx, row in forestloss_tiles.iterrows():
    print(idx)
    with rasterio.open(row.filename) as source:
        data = source.read(1)
    if not np.any(data == target_years[0]):
        continue
    
    tile_patches = 0
    for i in range(0, data.shape[0], patch_rows):
        #print(i)
        for j in range(0, data.shape[1], patch_cols):
            patch = data[i:i+patch_rows, j:j+patch_cols]

            forest_loss_mask = (np.isin(patch, target_years))

            # Check if there is enough forest loss
            if not forest_loss_mask.sum() >= min_px_forest_loss:
                continue
            tile_patches += 1
    
    patches_per_tile_min5pix.append(tile_patches)

0
opened
0
33
66
99
132
165
198
231
264
297
330
363
396
429
462
495
528
561
594
627
660
693
726
759
792
825
858
891
924
957
990
1023
1056
1089
1122
1155
1188
1221
1254
1287
1320
1353
1386
1419
1452
1485
1518
1551
1584
1617
1650
1683
1716
1749
1782
1815
1848
1881
1914
1947
1980
2013
2046
2079
2112
2145
2178
2211
2244
2277
2310
2343
2376
2409
2442
2475
2508
2541
2574
2607
2640
2673
2706
2739
2772
2805
2838
2871
2904
2937
2970
3003
3036
3069
3102
3135
3168
3201
3234
3267
3300
3333
3366
3399
3432
3465
3498
3531
3564
3597
3630
3663
3696
3729
3762
3795
3828
3861
3894
3927
3960
3993
4026
4059
4092
4125
4158
4191
4224
4257
4290
4323
4356
4389
4422
4455
4488
4521
4554
4587
4620
4653
4686
4719
4752
4785
4818
4851
4884
4917
4950
4983
5016
5049
5082
5115
5148
5181
5214
5247
5280
5313
5346
5379
5412
5445
5478
5511
5544
5577
5610
5643
5676
5709
5742
5775
5808
5841
5874
5907
5940
5973
6006
6039
6072
6105
6138
6171
6204
6237
6270
6303
6336
6369
6402
6435
6468
6501
6534
6567
6600
6633
6666
6699
6732
67

11319
11352
11385
11418
11451
11484
11517
11550
11583
11616
11649
11682
11715
11748
11781
11814
11847
11880
11913
11946
11979
12012
12045
12078
12111
12144
12177
12210
12243
12276
12309
12342
12375
12408
12441
12474
12507
12540
12573
12606
12639
12672
12705
12738
12771
12804
12837
12870
12903
12936
12969
13002
13035
13068
13101
13134
13167
13200
13233
13266
13299
13332
13365
13398
13431
13464
13497
13530
13563
13596
13629
13662
13695
13728
13761
13794
13827
13860
13893
13926
13959
13992
14025
14058
14091
14124
14157
14190
14223
14256
14289
14322
14355
14388
14421
14454
14487
14520
14553
14586
14619
14652
14685
14718
14751
14784
14817
14850
14883
14916
14949
14982
15015
15048
15081
15114
15147
15180
15213
15246
15279
15312
15345
15378
15411
15444
15477
15510
15543
15576
15609
15642
15675
15708
15741
15774
15807
15840
15873
15906
15939
15972
16005
16038
16071
16104
16137
16170
16203
16236
16269
16302
16335
16368
16401
16434
16467
16500
16533
16566
16599
16632
16665
16698
16731
16764
1679

KeyboardInterrupt: 

In [11]:
patches_per_tile_min5pix

[4898, 5732]

In [ ]:
forestloss_tiles['n_patches_min_5_px'] = patches_per_tile_min5pix

In [ ]:
forestloss_tiles['n_patches_min_10_px'] = patches_per_tile_min5pix

In [ ]:
print(patches_per_tile_min5pix)

In [33]:
forestloss_tiles['n_patches_min_15_px'] = patches_per_tile

In [ ]:
forestloss_tiles

In [ ]:
forestloss_tiles.n_patches_min_10_px.sum()

In [21]:
n_patches = 0



In [15]:
patch.shape

(33, 33)

In [17]:
patch.max()

0

In [22]:
n_patches

2572